In [ ]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path

if os.path.exists('/workspace/data'):
    DATA_DIR      = Path('/workspace/data')
    WORKSPACE_DIR = Path('/workspace')
elif os.path.exists('../environment/data'):
    DATA_DIR      = Path('../environment/data')
    WORKSPACE_DIR = Path('..')
elif os.path.exists('environment/data'):
    DATA_DIR      = Path('environment/data')
    WORKSPACE_DIR = Path('.')
else:
    DATA_DIR      = Path('data')
    WORKSPACE_DIR = Path('.')

In [ ]:
expenses_raw = pd.read_csv(DATA_DIR / 'expenses.csv',
                            parse_dates=['expense_date', 'payment_date'])
budgets      = pd.read_csv(DATA_DIR / 'budgets.csv')
departments  = pd.read_csv(DATA_DIR / 'departments.csv')
categories   = pd.read_csv(DATA_DIR / 'categories.csv')

In [ ]:
# Count unapproved rows before any filtering (raw file)
unapproved_expense_count = int((~expenses_raw['is_approved']).sum())

In [ ]:
# Keep only approved expenses; derive budget period from expense_date (accrual basis)
expenses = expenses_raw[expenses_raw['is_approved']].copy()
expenses['budget_month'] = expenses['expense_date'].dt.to_period('M').astype(str)

In [ ]:
# Prorate annual-contract categories to a monthly budget figure.
# software_licenses and professional_services store the full annual value;
# all other categories store the monthly allocation.
ANNUAL_CATS = {'software_licenses', 'professional_services'}

cat_name_map = dict(zip(categories['category_id'], categories['category_name']))

budgets = budgets.copy()
budgets['category_name'] = budgets['category_id'].map(cat_name_map)
budgets['monthly_budget_usd'] = np.where(
    budgets['category_name'].isin(ANNUAL_CATS),
    np.round(budgets['budgeted_amount'] / 12, 2),
    budgets['budgeted_amount']
)

In [ ]:
# Aggregate approved spend by department, category, and budget month
actual = (
    expenses
    .groupby(['department_id', 'category_id', 'budget_month'])['amount_usd']
    .sum()
    .reset_index()
    .rename(columns={'amount_usd': 'actual_spend_usd'})
)

In [ ]:
# Outer-join so every budgeted combination appears even with zero actuals
report = budgets.merge(
    actual,
    on=['department_id', 'category_id', 'budget_month'],
    how='outer'
)
report['actual_spend_usd']   = report['actual_spend_usd'].fillna(0.0)
report['monthly_budget_usd'] = report['monthly_budget_usd'].fillna(0.0)

In [ ]:
report['variance_usd'] = report['actual_spend_usd'] - report['monthly_budget_usd']
report['variance_pct'] = np.where(
    report['monthly_budget_usd'] > 0,
    np.round(report['variance_usd'] / report['monthly_budget_usd'] * 100, 2),
    None
)

In [ ]:
# Add dimension names and finalise column order
report = report.merge(departments[['department_id', 'department_name']],
                      on='department_id', how='left')
# category_name already present from budgets; drop duplicate if merge adds _x/_y
if 'category_name_y' in report.columns:
    report.drop(columns=[c for c in report.columns if c.endswith('_y')], inplace=True)
    report.rename(columns={c: c[:-2] for c in report.columns if c.endswith('_x')},
                  inplace=True)

report = report[[
    'department_id', 'department_name',
    'category_id',   'category_name',
    'budget_month',  'monthly_budget_usd',
    'actual_spend_usd', 'variance_usd', 'variance_pct'
]]
report = report.sort_values(['budget_month', 'department_id', 'category_id']).reset_index(drop=True)

In [ ]:
report.to_csv(WORKSPACE_DIR / 'variance_report.csv', index=False)

In [ ]:
total_budgeted_usd   = float(round(report['monthly_budget_usd'].sum(), 2))
total_actual_usd     = float(round(report['actual_spend_usd'].sum(), 2))
total_variance_usd   = float(round(total_actual_usd - total_budgeted_usd, 2))

dept_q1 = report.groupby('department_id').agg(
    q1_budget=('monthly_budget_usd', 'sum'),
    q1_actual=('actual_spend_usd',   'sum')
).reset_index()
over_budget_department_count = int((dept_q1['q1_actual'] > dept_q1['q1_budget']).sum())

sw_rows = report[report['category_name'] == 'software_licenses']
software_licenses_q1_budget_usd = float(round(sw_rows['monthly_budget_usd'].sum(), 2))

march_actual_spend = float(round(
    report.loc[report['budget_month'] == '2024-03', 'actual_spend_usd'].sum(), 2
))